# 02 Feature extraction — Random craton-edge points (target areas) — optimised mantle reference frame (Muller25_1.8_Ga_OMR_NNR_TPW)

Deposits and random craton-edge points are processed by **the same core code cell** (identical in all four notebooks; compare `CORE_VERSION`).

What changed relative to the previous version:

1. **Window direction.** For a sample of age *T* (Ma) the window is *T, T+1, …, T+20* Ma — the 20 Myr leading up to mineralisation. The old code used `time - year_offset`, i.e. the 20 Myr *after* the sample age.
2. **One value per sample.** Medians (`*_m`) and changes (`*_g`, `*_c`) are computed from each sample's own time series. Previously one value was computed per time step (random points) or per deposit type and age (deposits) and copied to every member.
3. **Same treatment for deposits and random points.** At every snapshot in the window, each sample is reconstructed with the rotation model (no interpolation), and subduction zones, craton segments and velocities are recomputed for that time.
4. **Change = value(T) − value(T+20).** Positive means the quantity increased towards the mineralisation age. Axial angles (`cr_ve_az`) are differenced on the circle, wrapped to ±90°.
5. **`ds_dp_tr`** (sample-to-trench distance) is computed at the sample age. Previously all samples used the oldest age in the dataset.
6. Distances are great-circle km; axial angles are averaged on the circle; `cr_ve_az` is an azimuth clockwise from north (previously `atan2(north, east)`).

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import gplately
from gplately import pygplates
print('gplately', gplately.__version__, '| pygplates', pygplates.__version__)

In [ ]:
# ------------------------------------------------------------------ PLATE MODEL
# Optimised mantle reference frame model: Muller25_1.8_Ga_OMR_NNR_TPW
MODEL_DIR = r'C:\Users\hshi9735\OneDrive - The University of Sydney (Students)\PhD_1_Notebooks\Deposits_Rev2\Muller25_1.8_Ga_OMR_NNR_TPW_ref_frame'
OMR_ROT_DIR = os.path.join(MODEL_DIR, 'optimisation')     # OMR + NNR + TPW rotation files
OMR_ANCHOR_PLATE_ID = 14                                   # as in the current notebooks

TOPOLOGY_FILES = [os.path.join(MODEL_DIR, f) for f in [
    '250-0_plate_boundaries.gpml',
    '410-250_plate_boundaries.gpml',
    '1800-1000_plate_boundaries.gpml',
    '1000-410-plate-boundaries.gpml',
    '1000-410-Convergence.gpml',
    '1000-410-Divergence.gpml',
    '1000-410-Transforms.gpml',
    'TopologyBuildingBlocks.gpml',
]]
STATIC_POLYGONS_FILE = os.path.join(MODEL_DIR, 'static_polygons.gpmlz')
CRATON_FILE = 'Inputs/merged_segmented_craton.shp'

# ------------------------------------------------------------------ WINDOW
WINDOW = 20            # Myr. Window = [T, T + 20] Ma, i.e. the 20 Myr LEADING UP TO the sample age T
STEP = 1               # Myr between reconstruction snapshots inside the window
MAX_TIME = 1800        # Ma, oldest time in the plate model; windows running past it -> NaN
TRENCH_RES_DEG = 0.5   # subduction-zone tessellation spacing (deg), as before
TRENCH_REF = 'craton'  # 'craton': sb_ve/tr_adv from trenches nearest the sample's nearest craton segment
                       # 'sample': from the trench point nearest the sample itself
N_JOBS = 20            # parallel workers (1 = serial)

# ------------------------------------------------------------------ DATA
INPUT_FILE = './Inputs/deposit_coords_recon_0.5_CSWM_CSCC.csv'   # random craton-edge points
USE_INPUT_PLATEIDS = False   # no plate-ID column: plate IDs from this model's static polygons

In [ ]:
rotation_files = sorted(glob.glob(os.path.join(OMR_ROT_DIR, '*.rot')))
assert rotation_files, f'No .rot files in {OMR_ROT_DIR}'
print('Rotation files (OMR):'); [print('  ', os.path.basename(f)) for f in rotation_files]
rotation_model = pygplates.RotationModel(rotation_files)

topology_features = pygplates.FeatureCollection(
    [f for fn in TOPOLOGY_FILES for f in pygplates.FeatureCollection(fn)])
static_polygons = pygplates.FeatureCollection(STATIC_POLYGONS_FILE)
craton_features = pygplates.FeatureCollection(CRATON_FILE)

model = gplately.PlateReconstruction(rotation_model=rotation_model,
                                     topology_features=topology_features,
                                     static_polygons=static_polygons,
                                     anchor_plate_id=OMR_ANCHOR_PLATE_ID)
ANCHOR_PLATE_ID = OMR_ANCHOR_PLATE_ID
print('Model loaded, anchor plate', ANCHOR_PLATE_ID, '| craton segments:', len(craton_features))

## Core code (identical in all four notebooks)

In [ ]:
# CORE_VERSION = '5575f5df58'  (same code cell in all four notebooks)
# =====================================================================
# CORE FEATURE-EXTRACTION CODE — identical in all four notebooks.
# Every sample (deposit or random craton-edge point) goes through the
# same functions: per-sample values, full plate reconstruction at every
# time step of the window, subduction zones and cratons re-reconstructed
# at every time step.
# =====================================================================
import warnings
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from joblib import Parallel, delayed
import gplately
from gplately import pygplates

EARTH_RADIUS_KM = 6371.0

# Convergence / trench-migration scaling kept from the original notebooks
# (gplately returns cm/yr; the original multiplied by 1e-2).
SB_SCALE = 1e-2

# Per-time "state" variables computed for every sample at every time step.
STATE_VARS = ['lat', 'lon', 'dp_ve', 'cr_ve', 'cr_ve_az', 'sb_ve',
              'tr_adv', 'tr_cr_an', 'ds_cr_tr', 'ds_dp_tr']
SV = {v: i for i, v in enumerate(STATE_VARS)}

# Instantaneous columns (value at the mineralisation / sample age T)
INSTANT_COLS = ['cr_ve', 'dp_ve', 'ds_cr_tr', 'sb_ve', 'tr_adv', 'tr_cr_an',
                'lat', 'lon', 'ds_dp_tr']
# Median over the window [T, T + WINDOW]   (state var -> output column)
MEDIAN_COLS = {'dp_ve': 'dp_ve_m', 'cr_ve': 'cr_ve_m', 'cr_ve_az': 'cr_ve_az_m',
               'tr_cr_an': 'tr_cr_an_m', 'sb_ve': 'sb_ve_m', 'tr_adv': 'tr_adv_m'}
# Change over the window: value(T) - value(T + WINDOW)
CHANGE_COLS = {'dp_ve': 'dp_ve_g', 'cr_ve': 'cr_ve_g', 'cr_ve_az': 'cr_ve_az_c',
               'tr_cr_an': 'tr_cr_an_c', 'sb_ve': 'sb_ve_g', 'tr_adv': 'tr_adv_g'}
AXIAL_VARS = {'cr_ve_az'}   # 0-180 axial angles: differences wrapped to [-90, 90)


# ---------------------------------------------------------------- geometry
def lonlat2xyz(lon, lat):
    lon = np.radians(np.asarray(lon, float))
    lat = np.radians(np.asarray(lat, float))
    return np.c_[np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)]


def chord_to_km(chord):
    """Unit-sphere chord length -> great-circle distance in km."""
    return 2.0 * np.arcsin(np.clip(np.asarray(chord) / 2.0, 0.0, 1.0)) * EARTH_RADIUS_KM


def axial_mean(deg):
    """Mean of axial (0-180) angles, handling the 0/180 wrap."""
    deg = np.asarray(deg, float)
    deg = deg[np.isfinite(deg)]
    if deg.size == 0:
        return np.nan
    a = np.radians(2.0 * deg)
    return (np.degrees(np.arctan2(np.sin(a).mean(), np.cos(a).mean())) / 2.0) % 180.0


def axial_diff(a, b):
    """a - b for axial angles, wrapped to [-90, 90)."""
    return (np.asarray(a) - np.asarray(b) + 90.0) % 180.0 - 90.0


def obliquity(a, b):
    """Smallest angle (0-90) between two axial directions."""
    d = np.abs(np.asarray(a, float) - np.asarray(b, float)) % 180.0
    return np.where(d > 90.0, 180.0 - d, d)


# ---------------------------------------------------------- plate IDs
def assign_plate_ids(lons, lats, static_polygons, rotation_model):
    """Plate ID and static-polygon begin age for each present-day point.
    Unassigned points get plate ID 0 (they are left as NaN downstream)."""
    partitioner = pygplates.PlatePartitioner(static_polygons, rotation_model,
                                             reconstruction_time=0.0)
    n = len(lons)
    pids = np.zeros(n, dtype=int)
    begin = np.full(n, np.nan)
    for i, (lo, la) in enumerate(zip(lons, lats)):
        rfg = partitioner.partition_point(pygplates.PointOnSphere(float(la), float(lo)))
        if rfg is not None:
            f = rfg.get_feature()
            pids[i] = f.get_reconstruction_plate_id()
            begin[i] = float(f.get_valid_time()[0])
    return pids, begin


# ---------------------------------------------------------- reconstruction
def reconstruct_points(lons, lats, pids, time, rotation_model, anchor_plate_id):
    """Rigid reconstruction of present-day points to `time` using each point's
    plate ID (full rotation-model reconstruction, no interpolation)."""
    lons = np.asarray(lons, float); lats = np.asarray(lats, float)
    pids = np.asarray(pids, int)
    out_lon = np.full(lons.size, np.nan); out_lat = np.full(lons.size, np.nan)
    for pid in np.unique(pids):
        if pid == 0:
            continue
        m = pids == pid
        rot = rotation_model.get_rotation(float(time), int(pid), 0.0,
                                          anchor_plate_id=int(anchor_plate_id),
                                          use_identity_for_missing_plate_ids=False)
        if rot is None:
            continue
        mp = pygplates.MultiPointOnSphere(list(zip(lats[m], lons[m])))
        ll = (rot * mp).to_lat_lon_array()
        out_lat[m] = ll[:, 0]; out_lon[m] = ll[:, 1]
    return out_lon, out_lat


def point_velocities(model, lons, lats, time):
    """Velocity magnitude (km/Myr = mm/yr) and azimuth (deg clockwise from N)."""
    lons = np.asarray(lons, float); lats = np.asarray(lats, float)
    mag = np.full(lons.size, np.nan); az = np.full(lons.size, np.nan)
    ok = np.isfinite(lons) & np.isfinite(lats)
    if not ok.any():
        return mag, az
    try:
        v = model.get_point_velocities(lons[ok], lats[ok], time,
                                       velocity_units=pygplates.VelocityUnits.kms_per_my)
    except TypeError:            # older gplately without velocity_units
        v = model.get_point_velocities(lons[ok], lats[ok], time)
    v = np.asarray(v, float)     # columns: (north, east)
    vn, ve = v[:, 0], v[:, 1]
    mag[ok] = np.hypot(vn, ve)
    az[ok] = np.degrees(np.arctan2(ve, vn)) % 360.0
    return mag, az


def tessellate_trenches(model, time, res_deg):
    try:
        sd = model.tessellate_subduction_zones(time, np.deg2rad(res_deg), ignore_warnings=True)
    except Exception:
        return None
    sd = np.asarray(sd, float)
    if sd.ndim != 2 or sd.shape[0] == 0:
        return None
    norm = sd[:, 7] % 360.0
    norm[norm >= 180.0] -= 180.0                       # trench normal as an axis
    xyz = lonlat2xyz(sd[:, 0], sd[:, 1])
    return dict(
        tree=cKDTree(xyz),
        conv=np.fabs(sd[:, 2]) * SB_SCALE * np.cos(np.radians(sd[:, 3])),
        mig=np.fabs(sd[:, 4]) * SB_SCALE * np.cos(np.radians(sd[:, 5])),
        norm=norm,
    )


def reconstruct_cratons(model, craton_features, time):
    """Vertices (lat, lon), owner index and centroids of craton segments at `time`."""
    rfgs = model.reconstruct(craton_features, time)
    verts, owner, centroids = [], [], []
    for rfg in (rfgs or []):
        geom = rfg.get_reconstructed_geometry()
        ll = geom.to_lat_lon_array()
        if ll.shape[0] == 0:
            continue
        verts.append(ll); owner.append(np.full(ll.shape[0], len(centroids)))
        centroids.append(geom.get_centroid().to_lat_lon())
    if not centroids:
        return None
    return dict(V=np.vstack(verts), own=np.concatenate(owner), centroids=np.asarray(centroids))


CRATON_PROPS = ['cr_ve', 'cr_ve_az', 'sb_ve', 'tr_adv', 'tr_cr_an', 'ds_cr_tr']


def craton_properties(cr, v_mag, v_az, trench):
    """One row of CRATON_PROPS per craton segment (vertex velocities supplied)."""
    V, own = cr['V'], cr['own']
    nk = len(cr['centroids'])
    vn = v_mag * np.cos(np.radians(v_az)); ve = v_mag * np.sin(np.radians(v_az))
    if trench is not None:
        d, j = trench['tree'].query(lonlat2xyz(V[:, 1], V[:, 0]))
        d_km = chord_to_km(d)
    props = np.full((nk, len(CRATON_PROPS)), np.nan)
    order = np.argsort(own, kind='stable')
    bounds = np.searchsorted(own[order], np.arange(nk + 1))
    for k in range(nk):
        m = order[bounds[k]:bounds[k + 1]]
        ll = V[m]
        if ll.shape[0] >= 2:          # segment orientation (axial 0-180)
            dlat = np.diff(ll[:, 0])
            dlon = (np.diff(ll[:, 1]) + 180.0) % 360.0 - 180.0
            cr_orient = axial_mean(np.degrees(np.arctan2(dlon, dlat)) % 180.0)
        else:
            cr_orient = np.nan
        props[k, 0] = np.nanmean(v_mag[m])                                   # cr_ve
        props[k, 1] = np.degrees(np.arctan2(np.nanmean(ve[m]),
                                            np.nanmean(vn[m]))) % 180.0      # cr_ve_az
        if trench is not None:
            props[k, 2] = trench['conv'][j[m]].mean()                        # sb_ve
            props[k, 3] = trench['mig'][j[m]].mean()                         # tr_adv
            props[k, 4] = obliquity(axial_mean(trench['norm'][j[m]]), cr_orient)  # tr_cr_an
            props[k, 5] = d_km[m].mean()                                     # ds_cr_tr
    return props


def state_at_time(time, lons0, lats0, pids, model, rotation_model, anchor_plate_id,
                  craton_features, trench_res_deg=0.5, trench_ref='craton'):
    """Per-sample state (STATE_VARS) for samples reconstructed to `time`.
    trench_ref='craton': sb_ve / tr_adv from trench points nearest the sample's
    nearest craton segment (as the original instantaneous values);
    trench_ref='sample': from the trench point nearest the sample itself."""
    n = len(lons0)
    S = np.full((n, len(STATE_VARS)), np.nan, dtype=np.float32)
    rlon, rlat = reconstruct_points(lons0, lats0, pids, time, rotation_model, anchor_plate_id)
    S[:, SV['lat']] = rlat; S[:, SV['lon']] = rlon
    ok = np.isfinite(rlon) & np.isfinite(rlat)
    if not ok.any():
        return S
    xyz = lonlat2xyz(rlon[ok], rlat[ok])
    nok = int(ok.sum())

    trench = tessellate_trenches(model, time, trench_res_deg)
    cr = reconstruct_cratons(model, craton_features, time)

    # one velocity call for samples + craton vertices (topologies resolved once)
    q_lon, q_lat = rlon[ok], rlat[ok]
    if cr is not None:
        q_lon = np.r_[q_lon, cr['V'][:, 1]]; q_lat = np.r_[q_lat, cr['V'][:, 0]]
    v_mag, v_az = point_velocities(model, q_lon, q_lat, time)
    S[ok, SV['dp_ve']] = v_mag[:nok]

    if trench is not None:
        d, j = trench['tree'].query(xyz)
        S[ok, SV['ds_dp_tr']] = chord_to_km(d)

    if cr is not None:
        props = craton_properties(cr, v_mag[nok:], v_az[nok:], trench)
        k = cKDTree(lonlat2xyz(cr['centroids'][:, 1], cr['centroids'][:, 0])).query(xyz)[1]
        for c, name in enumerate(CRATON_PROPS):
            S[ok, SV[name]] = props[k, c]
    if trench is not None and trench_ref == 'sample':
        S[ok, SV['sb_ve']] = trench['conv'][j]
        S[ok, SV['tr_adv']] = trench['mig'][j]
    return S


# ---------------------------------------------------------- driver
def _run_time_chunk(time_chunk, tasks, lons0, lats0, pids, model, rotation_model,
                    anchor_plate_id, craton_features, trench_res_deg, trench_ref):
    out = []
    for t in time_chunk:
        idx, kpos = tasks[t]
        S = state_at_time(t, lons0[idx], lats0[idx], pids[idx], model, rotation_model,
                          anchor_plate_id, craton_features, trench_res_deg, trench_ref)
        out.append((idx, kpos, S))
    return out


def compute_window_states(lons0, lats0, pids, ages, model, rotation_model, anchor_plate_id,
                          craton_features, window=20, step=1, max_time=1800,
                          trench_res_deg=0.5, trench_ref='craton', n_jobs=1, n_chunks=None, verbose=5):
    """States for every sample at times T, T+step, ..., T+window, where T is the
    sample age in Ma. Larger times are OLDER, so this is the window leading UP TO
    the sample age. Returns (states [n_samples, n_offsets, n_vars], offsets, T)."""
    lons0 = np.asarray(lons0, float); lats0 = np.asarray(lats0, float)
    pids = np.asarray(pids, int)
    T = np.rint(np.asarray(ages, float)).astype(int)
    offsets = np.arange(0, window + 1, step)
    states = np.full((len(T), len(offsets), len(STATE_VARS)), np.nan, dtype=np.float32)

    by_age = pd.Series(np.arange(len(T))).groupby(T).apply(np.asarray).to_dict()
    tasks = {}
    for age, idx in by_age.items():
        for kpos, k in enumerate(offsets):
            t = int(age + k)
            if t > max_time:
                continue
            a, b = tasks.get(t, ([], []))
            a.append(idx); b.append(np.full(idx.size, kpos))
            tasks[t] = (a, b)
    tasks = {t: (np.concatenate(a), np.concatenate(b)) for t, (a, b) in tasks.items()}
    times = sorted(tasks)
    print(f"{len(T)} samples, {len(by_age)} unique ages, {len(times)} reconstruction times "
          f"(window {window} Myr before each age, step {step} Myr)")

    if n_jobs == 1:
        chunks = [[t] for t in times]
        results = []
        for i, ch in enumerate(chunks):
            results.append(_run_time_chunk(ch, tasks, lons0, lats0, pids, model, rotation_model,
                                           anchor_plate_id, craton_features, trench_res_deg, trench_ref))
            if verbose and (i % 50 == 0 or i == len(chunks) - 1):
                print(f"  time {ch[0]} Ma  ({i + 1}/{len(chunks)})")
    else:
        n_chunks = n_chunks or max(1, 2 * abs(n_jobs))
        chunks = [list(c) for c in np.array_split(np.array(times), n_chunks) if len(c)]
        results = Parallel(n_jobs=n_jobs, verbose=verbose)(
            delayed(_run_time_chunk)(ch, {t: tasks[t] for t in ch}, lons0, lats0, pids, model,
                                     rotation_model, anchor_plate_id, craton_features,
                                     trench_res_deg, trench_ref)
            for ch in chunks)
    for chunk_res in results:
        for idx, kpos, S in chunk_res:
            states[idx, kpos, :] = S
    return states, offsets, T


def summarise_states(states, require_full_window=True):
    """Instantaneous, window-median and window-change columns, one value per sample."""
    out = {}
    s0 = states[:, 0, :]          # at sample age T
    sW = states[:, -1, :]         # at T + window (older)
    for c in INSTANT_COLS:
        out[c] = s0[:, SV[c]]
    complete = np.all(np.isfinite(states[:, :, SV['lat']]), axis=1)
    for v, c in MEDIAN_COLS.items():
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            med = np.nanmedian(states[:, :, SV[v]], axis=1)
        if require_full_window:
            med = np.where(complete, med, np.nan)
        out[c] = med
    for v, c in CHANGE_COLS.items():
        a, b = s0[:, SV[v]], sW[:, SV[v]]
        out[c] = axial_diff(a, b) if v in AXIAL_VARS else a - b
    out['window_complete'] = complete
    return pd.DataFrame(out)

## Load samples

In [ ]:
raw = pd.read_csv(INPUT_FILE)
raw['src_row'] = np.arange(len(raw))
df = raw[raw['age (Ma)'].notna() & (raw['age (Ma)'] > 0) & (raw['age (Ma)'] <= 1800)].copy()
df = df.reset_index(drop=True)
LON_COL, LAT_COL, AGE_COL = 'present_lon', 'present_lat', 'age (Ma)'
print(f'{len(df)} random points after filtering (of {len(raw)})')
print(f"Memory for window states: ~{len(df) * (WINDOW // STEP + 1) * 10 * 4 / 1e9:.2f} GB")

## Plate IDs (static polygons of the OMR model)

In [ ]:
pids, sp_begin = assign_plate_ids(df[LON_COL].values, df[LAT_COL].values, static_polygons, rotation_model)
if USE_INPUT_PLATEIDS and 'PLATEID1' in df.columns:
    print('Plate IDs differing from static-polygon assignment:',
          int((df['PLATEID1'].astype(int).values != pids).sum()))
    pids = df['PLATEID1'].astype(int).values
elif 'PLATEID1' in df.columns:
    print('Note: PLATEID1 in the input differs from the static-polygon plate ID for',
          int((df['PLATEID1'].astype(int).values != pids).sum()), 'samples (static-polygon IDs used)')
df['plate_id_used'] = pids
df['sp_begin_age'] = sp_begin
df['age_older_than_polygon'] = df[AGE_COL].values > sp_begin
print('Unassigned (plate ID 0, will be NaN):', int((pids == 0).sum()))
print('Sample age older than its static polygon:', int(df['age_older_than_polygon'].sum()))

## Per-sample states over the window [T, T+20] Ma

In [ ]:
states, offsets, T = compute_window_states(
    df[LON_COL].values, df[LAT_COL].values, df['plate_id_used'].values, df[AGE_COL].values,
    model, rotation_model, ANCHOR_PLATE_ID, craton_features,
    window=WINDOW, step=STEP, max_time=MAX_TIME, trench_res_deg=TRENCH_RES_DEG,
    trench_ref=TRENCH_REF, n_jobs=N_JOBS)

## Instantaneous, median and change features → CSV

In [ ]:
features = summarise_states(states, require_full_window=True)
df = df.drop(columns=[c for c in features.columns if c in df.columns])
df = pd.concat([df, features], axis=1)
df['recon_time'] = T

os.makedirs('./Outputs', exist_ok=True)
OUTPUT_FILE = './Outputs/deposit_data_recon_0.5_CSWM_CSCC_OMR.csv'
df.to_csv(OUTPUT_FILE, index=False)
print('Saved', OUTPUT_FILE, df.shape)
print('Incomplete windows (NaN medians/changes):', int((~df['window_complete']).sum()))

SAVE_WINDOW_STATES = False   # full per-sample time series, for auditing
if SAVE_WINDOW_STATES:
    np.savez_compressed('./Outputs/Random_window_states_OMR.npz', states=states, offsets=offsets, T=T,
                        state_vars=np.array(STATE_VARS), src_row=df['src_row'].values)
df.head()

## Sanity checks

In [ ]:
# ------------------------------------------------------------------ SANITY CHECKS
rng = np.random.default_rng(1)

# 1) the window runs back in time: snapshot k is at T + k (older), never T - k
print('Window offsets (Myr added to sample age):', offsets[0], '...', offsets[-1])
assert offsets[-1] == WINDOW and np.all(np.diff(offsets) > 0)

# 2) one value per sample, not per group: spread of each change variable within an age
grp = df.groupby('recon_time')
multi = grp.size() > 1
for c in CHANGE_COLS.values():
    nuniq = grp[c].nunique()[multi]
    print(f'{c:12s} ages with >1 sample: {multi.sum():5d} | of these, ages with >1 distinct value: {(nuniq > 1).sum():5d}')
print('(craton-derived values are shared only by samples whose nearest craton segment is the same)')

# 3) independent recomputation for random samples at T and at T + WINDOW
ok = np.where(df['window_complete'].values)[0]
for i in rng.choice(ok, size=min(5, ok.size), replace=False):
    r = df.iloc[i]
    s_T = state_at_time(int(T[i]), [r[LON_COL]], [r[LAT_COL]], [r['plate_id_used']], model,
                        rotation_model, ANCHOR_PLATE_ID, craton_features, TRENCH_RES_DEG, TRENCH_REF)[0]
    s_W = state_at_time(int(T[i]) + WINDOW, [r[LON_COL]], [r[LAT_COL]], [r['plate_id_used']], model,
                        rotation_model, ANCHOR_PLATE_ID, craton_features, TRENCH_RES_DEG, TRENCH_REF)[0]
    expect = s_T[SV['sb_ve']] - s_W[SV['sb_ve']]
    print(f"row {i}: T={T[i]} Ma  sb_ve_g stored={r['sb_ve_g']:.5f}  recomputed={expect:.5f}")
    assert np.isclose(r['sb_ve_g'], expect, equal_nan=True, atol=1e-5)

# 4) rigid reconstruction agrees with gplately.Points for a few samples
j = rng.choice(ok, size=min(5, ok.size), replace=False)
for i in j:
    p = gplately.Points(model, [df[LON_COL].iloc[i]], [df[LAT_COL].iloc[i]])
    lo, la = p.reconstruct(int(T[i]), return_array=True)
    print(f'row {i}: lat {df.lat.iloc[i]:.4f} vs gplately.Points {float(np.ravel(la)[0]):.4f}')
print('All checks passed.')